# Disease Prediction Model Training
This notebook contains the pipeline to preprocess the symptoms dataset, generate synthetic data, handle class imbalance using SMOTE, train a Random Forest classifier, and run SHAP explainability analyses.

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE
import shap

# Set paths relative to notebooks directory
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")
MODELS_DIR = os.path.join(BASE_DIR, "app", "models")
RAW_DATA_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")

In [ ]:
# Load and preprocess data
csv_path = os.path.join(RAW_DATA_DIR, "Training.csv")
df = pd.read_csv(csv_path)

if 'Unnamed: 133' in df.columns:
    df = df.drop(columns=['Unnamed: 133'])
    
symptom_columns = [col for col in df.columns if col != 'prognosis']
df['prognosis'] = df['prognosis'].str.strip()

label_mapping = {
    "Fungal infection": "Fungal Skin Infection",
    "Allergic rhinitis": "Allergic Rhinitis",
    "GERD": "Gastritis",
    "Peptic ulcer diseae": "Gastritis",
    "Gastroenteritis": "Food Poisoning",
    "Diabetes": "Diabetes",
    "Diabetes ": "Diabetes",
    "Hypertension": "Hypertension",
    "Hypertension ": "Hypertension",
    "Heart attack": "Coronary Heart Disease",
    "Bronchial Asthma": "Asthma",
    "Pneumonia": "Pneumonia",
    "Tuberculosis": "Tuberculosis (TB)",
    "Jaundice": "Jaundice (Viral Hepatitis A/E)",
    "hepatitis A": "Jaundice (Viral Hepatitis A/E)",
    "Hepatitis E": "Jaundice (Viral Hepatitis A/E)",
    "Common Cold": "Common Cold",
    "Migraine": "Migraine",
    "Typhoid": "Typhoid",
    "Dengue": "Dengue",
    "Malaria": "Malaria"
}

df['disease_mapped'] = df['prognosis'].map(label_mapping)
filtered_df = df[df['disease_mapped'].notna()].copy()
filtered_df = filtered_df.drop(columns=['prognosis']).rename(columns={'disease_mapped': 'prognosis'})

In [ ]:
# Synthesize data for missing conditions
missing_diseases_symptoms = {
    "Influenza": ["high_fever", "cough", "chills", "headache", "throat_irritation", "muscle_pain", "runny_nose", "fatigue", "malaise"],
    "COVID-19": ["high_fever", "cough", "breathlessness", "fatigue", "muscle_pain", "throat_irritation", "loss_of_appetite", "chest_pain"],
    "Chikungunya": ["high_fever", "joint_pain", "muscle_pain", "headache", "skin_rash", "fatigue"],
    "Bronchitis": ["cough", "breathlessness", "phlegm", "chest_pain", "fatigue"],
    "Anemia (Iron-deficiency)": ["fatigue", "lethargy", "dizziness", "headache", "breathlessness", "weakness_in_limbs"],
    "Vitamin D Deficiency": ["fatigue", "muscle_pain", "joint_pain"]
}

synthetic_rows = []
np.random.seed(42)

for disease, key_symptoms in missing_diseases_symptoms.items():
    for _ in range(150):
        row = {sym: 0 for sym in symptom_columns}
        for sym in key_symptoms:
            if sym in row:
                row[sym] = 1 if np.random.rand() > 0.15 else 0
        if sum(row.values()) == 0:
            row[key_symptoms[0]] = 1
        row['prognosis'] = disease
        synthetic_rows.append(row)
        
synthetic_df = pd.DataFrame(synthetic_rows)
final_df = pd.concat([filtered_df, synthetic_df], ignore_index=True)

# Save symptoms list to processed folder
symptom_list_path = os.path.join(PROCESSED_DATA_DIR, "symptoms.json")
with open(symptom_list_path, "w") as f:
    json.dump(symptom_columns, f, indent=4)

In [ ]:
# Train Random Forest Classifier
X = final_df[symptom_columns]
y = final_df['prognosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_res, y_train_res)

rf_preds = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, rf_preds):.4f}")

In [ ]:
# Save model and explainer
os.makedirs(MODELS_DIR, exist_ok=True)
with open(os.path.join(MODELS_DIR, "disease_model.pkl"), "wb") as f:
    pickle.dump(rf_model, f)
    
explainer = shap.TreeExplainer(rf_model)
with open(os.path.join(MODELS_DIR, "disease_shap_explainer.pkl"), "wb") as f:
    pickle.dump(explainer, f)
print("Training complete and models saved successfully!")